# 📘 Case 1: MRI Preprocessing Pipeline (DeepPrep)
## Scientific Workflows for Brain and Behavioral Research
### National Science Foundation Supported Learning Initiative (Award No. OAC-2417875)

> ## ⚠️ Optional Advanced Resource
>
> **This notebook is not required for the workshop.** It is an optional, advanced walkthrough for participants who work with MRI data and want to see a full research pipeline running on a real computing system.
>
> Unlike the rest of the pre-workshop notebooks, most of this notebook cannot run in Google Colab. It requires a real Ubuntu virtual machine (VM) with at least 16 GB of working memory (RAM) — for example, a computing session on FABRIC, which you practiced requesting in Pre-Workshop Notebook 3.

This notebook has **two parts**:

| Part | Focus | Where the code runs |
|------|-------|----------------------|
| **Planning (this Colab)** | Understanding the pipeline, verifying paths, estimating resources | Simulations in Colab → standard Python only |
| **Running the pipeline** | Installing Docker, downloading data, running DeepPrep | Real Ubuntu VM (a FABRIC computing session, CloudLab, or any cloud instance) |

**DeepPrep** is a deep-learning-accelerated pipeline for preparing structural and functional MRI data for analysis. It follows **BIDS** (the standard brain imaging data structure — a community convention for organizing MRI files so any tool can find them) and runs inside a **container** (a portable, self-contained package — built with the tool Docker — that includes everything the pipeline needs so it runs identically on any computer). This notebook preprocesses a paired **tumor** vs **control** resting-state dataset on a single VM — the same recipe scales to a larger research computing system.

***

## 📋 Set Up Your Profile
*Run the cell below to record your name and institution in this notebook.*

In [ ]:
# PROFILE SETUP
student_name = "Your Name"
institution = "Your University/Organization"
research_area = "Neuroscience / Psychology / Biology / Engineering / Computer Science / Other"

print(f"✅ Notebook initialized for {student_name} ({institution})")
print(f"Research area: {research_area}")

***
## 🤖 Using AI Tools to Build Your Workflow ("Ask → Build → Document")

As in the other notebooks, you act as a **Workflow Designer**: you decide what the pipeline needs to do, and use an AI tool to help generate supporting code, then document what you learned.

> The two activities below (marked 🧭) are planning simulations that run safely in Colab, using only the Python standard library.  
> Everything else in this notebook is a **reference walkthrough** of the real commands, marked 🪐. Those cells are commented out — they are meant to be studied here and run later on a real Ubuntu VM, not in Colab.

***
## 🧭 Roadmap

**Request a VM → Install Docker → Install the Hugging Face CLI → Download the data → Get a FreeSurfer license → Pull the DeepPrep image → Run TUMOR → Run CONTROL → Inspect outputs.**

***
## ⚠️ Prerequisites

1. **Download path vs. mount path.** The dataset downloads to `/home/ubuntu/tumor_control/`, and the pipeline-run commands expect the tumor and control MRI files at `/home/ubuntu/tumor_control/tumor/bids` and `/home/ubuntu/tumor_control/control/bids`. Before running the pipeline, always **verify** where the `bids/` folders actually landed — this is the single most common source of errors.
2. **Minimum resources** (per the DeepPrep documentation): Ubuntu 20.04 or newer, at least 16 GB of working memory plus swap space (64 GB is recommended for full runs), at least 20 GB of permanent storage (realistically more, once the container image, data, and outputs are included), and 12+ GB of graphics memory (VRAM) if using a graphics processor (GPU) to speed things up.

***
## 1 · 🖥️ Request and Connect to a VM

Request (or connect to) an Ubuntu 20.04-or-newer VM — a FABRIC computing session, a CloudLab node, or any cloud instance. You will need administrator access, outbound internet, and enough storage.

> 💡 **Recommended path:** if you are using FABRIC, **FABRIC JupyterHub Access & Setup** walks through provisioning this VM the way FABRIC itself recommends — inside FABRIC's own JupyterHub, with credentials and `fablib` already configured. Complete that first, then come back here once your session is Ready.

In [ ]:
# 🪐 Connect to your VM  (run only on a real system, not in Colab)
# Replace <management-address> with the actual management address of your computing session (from Notebook 3):
# ssh -F ~/.ssh/config -i slice_key ubuntu@<management-address>

## 2 · 🐳 Install Docker
DeepPrep ships as a Docker container image — Docker is the only *required* container tool for this pipeline.

[Docker install guide for Ubuntu](https://docs.docker.com/engine/install/ubuntu/)

In [ ]:
# 🪐 Install Docker  (run only on a real system, not in Colab)
# Official convenience script (Ubuntu). For production, prefer the apt-repo method in the guide above.
# curl -fsSL https://get.docker.com | sudo sh
#
# Verify the engine works:
# sudo docker run -it --rm hello-world

## 3 · 🤗 Install the Hugging Face Command-Line Tool (`hf`)
`hf` is the current Hugging Face command-line tool used to download the shared dataset. It ships inside the `huggingface_hub` package.

[hf CLI docs](https://huggingface.co/docs/huggingface_hub/main/en/guides/cli)

In [ ]:
# 🪐 Install the hf command-line tool  (run only on a real system, not in Colab)
# pip install -U 'huggingface_hub[hf_xet]'   # hf_xet = faster large-file transfers
# hf version

## 4 · ⬇️ Download the Tumor-Control Dataset
To keep the paths predictable, download into `/home/ubuntu/tumor_control` and then **verify** where the `bids/` folders actually landed.

In [ ]:
# 🪐 Download and verify the dataset  (run only on a real system, not in Colab)
# hf download xenificity/tumor_control --repo-type dataset --local-dir /home/ubuntu/tumor_control
#
# >>> VERIFY before running DeepPrep <<<  find the real BIDS folders:
# find /home/ubuntu/tumor_control -maxdepth 3 -type d -name bids
#
# You want these two paths to exist (they are what the docker -v mount flags below expect):
#   /home/ubuntu/tumor_control/tumor/bids
#   /home/ubuntu/tumor_control/control/bids
# If find prints them, you are ready. If they are nested elsewhere, either re-download with the
# correct --local-dir, or edit the mount source paths in the run cells below.

## 5 · 🔑 Request a FreeSurfer License
DeepPrep uses FreeSurfer internally, which requires a **free** license file. Register, then save the emailed `license.txt` to `/home/ubuntu/freesurfer_key/license.txt` — the path the pipeline-run commands expect.

[Request a FreeSurfer license](https://surfer.nmr.mgh.harvard.edu/registration.html) (free; you receive the key by email).

In [ ]:
# 🪐 Place your FreeSurfer license  (run only on a real system, not in Colab)
# mkdir -p /home/ubuntu/freesurfer_key
#
# Option A: paste the license contents (5 lines from the email) into a heredoc:
# cat > /home/ubuntu/freesurfer_key/license.txt << 'EOF'
# your@email.edu
# 00000
#  *Cxxxxxxxxxxxx
#  FSxxxxxxxxxxxxx
# EOF
#
# Option B: if you already transferred license.txt some other way, move it into place:
# mv /path/to/license.txt /home/ubuntu/freesurfer_key/license.txt
#
# Confirm it exists and is not empty:
# test -s /home/ubuntu/freesurfer_key/license.txt && echo 'license.txt present' || echo 'MISSING license.txt'

## 6 · 📦 Pull the DeepPrep Image and Read Its Usage
Pull the container image once (a few GB). Running it with no arguments prints the full list of options.

[DeepPrep installation guide](https://deepprep.readthedocs.io/en/latest/installation.html)

In [ ]:
# 🪐 Pull the image and inspect its usage  (run only on a real system, not in Colab)
# sudo docker pull pbfslab/deepprep:25.1.0
#
# Print DeepPrep's own argument help:
# sudo docker run --rm pbfslab/deepprep:25.1.0
#
# Positional arguments are always:  [bids_dir] [output_dir] participant
# then flags: --fs_license_file, --bold_task_type, --cpus, --memory, --device, --anat_only, --bold_only, ...
#
# Create the shared output folder the runs below mount:
# mkdir -p /home/ubuntu/output

***
### 🧭 Activity 1 — Build a Path-Verification and Mount Helper

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the code cell beneath it. This activity runs safely in Colab — it is a planning simulation, not a real pipeline run.*
>
> "Act as a neuroimaging workflow instructor helping a student prepare a DeepPrep run on a remote Ubuntu VM. Write a pure-Python script that:
> 1. Accepts a base directory (default `/home/ubuntu/tumor_control`) and checks whether the two expected BIDS folders exist: `<base>/tumor/bids` and `<base>/control/bids`.
> 2. If they exist, prints the exact `docker run -v` mount lines a researcher should use.
> 3. If they do not exist, runs a simulated folder-search (you may hard-code a few plausible nested layouts) and suggests the corrected mount paths.
> 4. Also prints a short checklist: FreeSurfer license present? Output folder created? Docker running?
> 5. Use only the Python standard library (os, pathlib). Add plain-language comments that explain why path mismatches are the most common cause of an empty input folder inside the container."

In [ ]:
# 💻 Activity 1 — Paste your AI-generated Path-Verification & Mount Helper code here:

## 7 · 🧠 Run DeepPrep on the TUMOR Data
This is the **processor-only (CPU) run**: no graphics processor is requested, so DeepPrep runs on the regular processor. The stopped container is also kept (not automatically removed) so it can be inspected afterward. `--bold_task_type rest` requires the tumor MRI files to carry the `task-rest` label.

> Expect this to be **slow** without a graphics processor (hours per participant). Launch it inside a `tmux` or `screen` session so it survives a dropped connection.

In [ ]:
# 🪐 Run DeepPrep on the tumor data  (run only on a real system, not in Colab)
# (sanity check) confirm there is at least one task-rest scan file in the tumor data:
# find /home/ubuntu/tumor_control/tumor/bids -name '*task-rest*_bold.nii*' | head
#
# Processor-only run. Adjust --cpus/--memory to match your system.
# sudo docker run -it \
#   -v /home/ubuntu/tumor_control/tumor/bids:/input \
#   -v /home/ubuntu/output:/output \
#   -v /home/ubuntu/freesurfer_key/license.txt:/fs_license.txt \
#   pbfslab/deepprep:25.1.0 \
#   /input /output participant \
#   --fs_license_file /fs_license.txt \
#   --bold_task_type rest \
#   --cpus 16 --memory 64
#
# Clean up the stopped container afterwards:
#   sudo docker ps -a | grep deepprep      # find the container id/name
#   sudo docker rm <container_id>

## 8 · 🚀 Run DeepPrep on the CONTROL Data
This is the **graphics-processor (GPU) accelerated run**, which is much faster, and automatically removes the container when finished. It requires a working GPU and the NVIDIA Container Toolkit.

> Outputs go to the **same** output folder as the tumor run. Since tumor and control participants have different IDs, they can safely share one output tree. If you would rather keep them fully separate, point this run at a different output folder instead.

In [ ]:
# 🪐 Run DeepPrep on the control data  (run only on a real system, not in Colab)
# (sanity check) confirm control task-rest scan files exist:
# find /home/ubuntu/tumor_control/control/bids -name '*task-rest*_bold.nii*' | head
#
# GPU-accelerated run.
# sudo docker run -it \
#   --gpus all \
#   -v /home/ubuntu/tumor_control/control/bids:/input \
#   -v /home/ubuntu/output:/output \
#   -v /home/ubuntu/freesurfer_key/license.txt:/fs_license.txt \
#   pbfslab/deepprep:25.1.0 \
#   /input /output participant \
#   --fs_license_file /fs_license.txt \
#   --bold_task_type rest \
#   --cpus 16 --memory 64

## 9 · 📂 Inspect Outputs and Quality-Control Reports
DeepPrep writes its results in the standard BIDS-Derivatives layout, plus a quality-control (QC) report for each participant. Always look through these reports before trusting any later analysis — this matters especially for clinical or tumor scans, where alignment problems are most likely to appear.

In [ ]:
# 🪐 Inspect outputs  (run only on a real system, not in Colab)
# Top-level derivatives:
# ls -R /home/ubuntu/output | head -50
#
# Find the per-participant QC reports (open these in a browser or via JupyterHub's file view):
# find /home/ubuntu/output -name '*.html' | head
#
# Runtime / resource reports are also produced — useful for benchmarking:
# find /home/ubuntu/output -iname '*report*' -o -iname '*timeline*' 2>/dev/null | head

***
### 🧭 Activity 2 — Simulate a Lightweight Resource and QC Checklist

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the code cell beneath it. This activity runs safely in Colab — it is a planning simulation, not a real pipeline run.*
>
> "Act as a neuroimaging data-management instructor. Write a pure-Python script that helps a student prepare for a long DeepPrep run. The script should:
> 1. Accept (as variables) the number of participants, estimated hours per participant on the regular processor, available working memory (GB), and whether a graphics processor is present.
> 2. Print a simple resource plan: total estimated wall-clock time, recommended `--cpus` and `--memory` values, and a warning if working memory looks insufficient (below 16 GB).
> 3. Print a pre-flight checklist with checkboxes (use Unicode ☐/☑) covering: Docker installed, FreeSurfer license present, BIDS paths verified, tmux/screen session ready, output folder created.
> 4. End with a short plain-language reminder about why QC reports must be inspected before any group-level analysis, especially for tumor data.
> Use only the Python standard library."

In [ ]:
# 💻 Activity 2 — Paste your AI-generated Resource & QC Checklist code here:

***
## 📌 Reference — Key DeepPrep Flags

| Flag | Meaning |
|---|---|
| `[bids_dir] [output_dir] participant` | Positional: input BIDS folder, output folder, analysis level (always `participant`). |
| `--fs_license_file PATH` | Path (inside the container) to the FreeSurfer `license.txt`. |
| `--bold_task_type 'rest'` | The BIDS task label(s) to process; must match the file names. Multiple labels: `'rest task1 task2'`. |
| `--cpus N` / `--memory N` | Processor cores and working memory (in GB) made available to the pipeline. |
| `--device {auto\|0\|1\|cpu}` | Compute device. Default `auto` (uses a graphics processor if available, otherwise the regular processor). |
| `--participant_label '001 002'` | Restrict the run to specific participants. |
| `--anat_only` / `--bold_only` | Run only the structural or only the functional stream. |
| `--skip_bids_validation` | Bypass the BIDS format check (use only if you already trust the file layout). |
| `--resume` | Resume a previous run instead of starting over. |

> Docker-level flags are separate: `--gpus all` (graphics-processor access), `--rm` (automatically remove the container when done), `-v host:container` (folder mounts).

***
## 📌 Troubleshooting

| Symptom | Likely cause / fix |
|---|---|
| DeepPrep error: license invalid or not found | `license.txt` is missing, empty, or mounted at the wrong path. Re-check Section 5 and the license mount line. |
| The input folder looks empty inside the container | Path mismatch from Section 4. Re-run the folder-search check and fix the mount source path. |
| No matching functional scans found | The BIDS files do not use the `task-rest` label. Inspect file names and set `--bold_task_type` to the real label. |
| Job dies when the connection drops | A long run was not detached from the terminal. Use `tmux`/`screen`. |
| Out of memory / process killed | Lower `--cpus`/`--memory` to fit the node, or add swap space. A GPU run also needs at least 12 GB of graphics memory. |
| Stopped containers piling up | The tumor run does not auto-remove its container. Clean up with `sudo docker rm <id>`. |

***
### ✍️ Reflection
*Double-click this cell to write your response.*

* **What I observed:** Why is verifying the exact BIDS paths *before* launching a multi-hour pipeline run so important? What would happen if a mount pointed at the wrong folder?
* **Connecting to key concepts:** This notebook demonstrates containerization, BIDS organization, license management, and resource-aware workflow design. Explain in plain language why the FreeSurfer license file and the output folder should be treated as first-class parts of the experimental workflow, not afterthoughts.

***
## 📖 Reference Guide: Key Concepts for MRI Preprocessing Pipelines

1. **BIDS (Brain Imaging Data Structure)** – The standard brain imaging data structure; a community convention for organizing MRI files so any compatible tool can find them automatically.
2. **Containerization (Docker)** – Packages an entire pipeline (all its dependencies included) into a portable container so it runs identically on any host computer.
3. **Volume Mount (`-v host:container`)** – Makes a folder on the host computer visible inside the container; path mismatches are the most common cause of pipeline failures.
4. **FreeSurfer License** – A required free registration; the resulting license file must be mounted into the container at a known path.
5. **Workflow Manager (Nextflow)** – The tool that orchestrates the many internal stages of DeepPrep and provides progress and timeline reports.
6. **CPU vs. GPU Execution** – DeepPrep can run on the regular processor (slower) or a graphics processor (much faster); a GPU run needs at least 12 GB of graphics memory.
7. **Quality-Control (QC) Reports** – Per-participant HTML reports generated by DeepPrep; must be inspected before any group-level analysis, especially for clinical or tumor data.
8. **Resource-Aware Launch** – Matching `--cpus` / `--memory` to the real capacity of the computing session prevents crashes and wasted time.

***
## 🔗 External Resources & Further Study

| Resource | Link |
|----------|------|
| DeepPrep documentation | https://deepprep.readthedocs.io/ |
| DeepPrep installation guide | https://deepprep.readthedocs.io/en/latest/installation.html |
| DeepPrep Docker image | `pbfslab/deepprep:25.1.0` |
| FreeSurfer license request | https://surfer.nmr.mgh.harvard.edu/registration.html |
| BIDS specification | https://bids.neuroimaging.io/ |
| Hugging Face CLI (`hf`) docs | https://huggingface.co/docs/huggingface_hub/main/en/guides/cli |
| Docker install guide (Ubuntu) | https://docs.docker.com/engine/install/ubuntu/ |

***
*End of Case 1 – Running an MRI Preprocessing Pipeline (DeepPrep)*